> $\sigma(\cdot)$: logistic sigmoid

- $\sigma(z)=\frac{1}{1+\exp(-z)}$
- $\tanh(z)=\frac{\exp(z)-\exp(-z)}{\exp(z)+\exp(-z)}=2\sigma(2z)-1$
- $\text{SiLU}(z)=z\sigma(z)$：Sigmoid Linear Unit
    - 也叫 Swish: a Self-Gated Activation Function
    - https://docs.pytorch.org/docs/2.13/generated/torch.nn.functional.silu.html
        - $\text{silu}(x) = x * \sigma(x), \text{where } \sigma(x) \text{ is the logistic sigmoid.}$
            - $x$: linear, $\sigma(x)$: sigmoid gate
-----

- $\operatorname{GLU}(x)=(xW_1+b_1)\odot\sigma(xW_2+b_2)$
    - Gated linear unit
    - Transformer 中常见的 SwiGLU 则把 sigmoid 换成 SiLU/Swish：
    - $\operatorname{SwiGLU}(x)=\operatorname{SiLU}(xW_1)\odot(xW_2)$


$$
\begin{split}
\operatorname{SiTU\text{-}GLU}(x)&=\left[\operatorname{softcap}(\mathbf{W}_g x,\beta_1)\odot\operatorname{Sigmoid}(\mathbf{W}_g x)\right]\odot\operatorname{softcap}(\mathbf{W}_u x,\beta_2),\qquad \beta_1=4,\ \beta_2=25 \\
&=\underbrace{\left[\beta_1\tanh\left(\frac{\mathbf{W}_g x}{\beta_1}\right)\odot\operatorname{Sigmoid}(\mathbf{W}_g x)\right]}_{\text{gate 支路 }\phi_g}\odot\underbrace{\left[\beta_2\tanh\left(\frac{\mathbf{W}_u x}{\beta_2}\right)\right]}_{\text{up 支路 }\phi_u}
\end{split}
$$
- $\operatorname{softcap}(a,b)=b\tanh\left(\frac{a}{b}\right),\qquad b>0$
    - $a$ 是要被限制的值，$b$ softcap 的阈值
    - Gemma 2 对 attention logits 和最终 logits 使用的就是这种形式。
    - `softcapped_logits = softcap * torch.tanh(logits / softcap)`
-----

- $\alpha_{t,j}^h=\exp\!\left[g_{\min}\operatorname{Sigmoid}\!\left(e^{A_h}z_{t,j}^h\right)\right],\qquad g_{\min}=-5$
    - $t$: token 位置, $j$: key 通道索引 (1, ..., 128)
    - $g_{t,j}^h=g_{\min}\operatorname{Sigmoid}\!\left(e^{A_h}z_{t,j}^h\right)\in(-5,0)$
    - $\alpha_{t,j}^h=\exp(g_{t,j}^h)\in(e^{-5},1)$
- kimi linear
    - $g=-e^{A_h}\operatorname{Softplus}(z)\in(-\infty,0)$
        - Softplus(x)=log(1+e^x).
    - K3 改为直接约束对数衰减：$g\in(-5,0)$

先忽略偏置并取 $A_h=0$：

| $z$ | $g=-5\sigma(z)$ | $\alpha=e^g$ | 含义 |
|---:|---:|---:|---|
| $-4$ | $-0.090$ | $0.914$ | 大部分保留 |
| $0$ | $-2.5$ | $0.082$ | 强遗忘 |
| $4$ | $-4.91$ | $0.0074$ | 近似清空 |

衰减 logit $z$ 经过 KDA 的复合激活后，会被转换成旧记忆的一步保留比例 $\alpha$。$z$ 越大，旧记忆保留得越少。